# 第 5 课：Mel 与 Log-Mel

这一课回答：为什么不把 STFT 原样交给 ASR？Mel 到底是什么？三角滤波器做了什么？为什么最后还要取 log？

路线：人耳频率感知 → Hz 与 Mel → Mel 滤波器组 → 频谱压缩 → log 压缩 → 真实语音 Log-Mel。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 声音与声学特征 |
| 建议投入 | 2～4 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 4 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | Mel 标度、滤波器组、Log-Mel |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：Mel 标度、滤波器组、Log-Mel。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：幅度、功率与 dB 的不同公式；STFT shape；频率 bin 的 Hz 含义。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [音频基础 3](音频基础_03_振幅RMS功率与dB.ipynb)与主线第 4 课，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：非负频率功率谱 `[T,F]` 与采样率
  ↓ 本课要学会的变换、状态或判断
输出：Mel 滤波器组能量和 Log-Mel `[T,M]`
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import ipywidgets as widgets
from IPython.display import Audio, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

## 1. 人耳不是按线性 Hz 感知频率

从 100 Hz 增加到 200 Hz，与从 1000 Hz 增加到 1100 Hz，虽然都增加 100 Hz，主观变化并不相同。人耳通常更接近感知‘比例’：频率翻倍称为一个八度。

100→200、200→400、400→800 Hz 都是翻倍。在线性 Hz 轴上间距越来越大，但听觉上的音程关系相似。

In [ ]:
octaves = np.array([100,200,400,800,1600,3200,6400])
fig,axes=plt.subplots(2,1,figsize=(11,5))
axes[0].scatter(octaves,np.zeros_like(octaves),s=60)
axes[0].set_xscale('linear'); axes[0].set_title('线性 Hz 轴：翻倍点越来越远')
axes[1].scatter(octaves,np.zeros_like(octaves),s=60,color='tab:orange')
axes[1].set_xscale('log'); axes[1].set_title('对数频率轴：翻倍点间距接近相等')
for ax in axes:
    ax.set_xlabel('Frequency (Hz)'); ax.set_yticks([]); ax.grid(axis='x',alpha=0.25)
    for f in octaves: ax.annotate(str(f),(f,0),xytext=(0,10),textcoords='offset points',ha='center')
plt.tight_layout(); plt.show()

### 热身题

1. 300 Hz 翻倍是多少 Hz？
2. 1000→2000 Hz 和 2000→4000 Hz，各跨越几个八度？
3. 为什么线性 Hz 刻度不完全符合人耳的频率分辨方式？

## 2. Hz 与 Mel 的转换

常见公式：

$$m=2595\log_{10}(1+f/700)$$

Mel 不是麦克风测出的新物理频率，而是一种重新标记频率位置的感知尺度。低频区域展开得更细，高频区域压缩得更多。

In [ ]:
def hz_to_mel(hz):
    return 2595*np.log10(1+np.asarray(hz)/700)

def mel_to_hz(mel):
    return 700*(10**(np.asarray(mel)/2595)-1)

hz=np.linspace(0,8000,1000)
plt.figure(figsize=(10,4))
plt.plot(hz,hz_to_mel(hz))
plt.xlabel('Frequency (Hz)'); plt.ylabel('Mel')
plt.title('Hz → Mel：越到高频，曲线增长越慢')
plt.grid(alpha=0.25); plt.show()

In [ ]:
def mel_converter(hz=1000):
    mel=float(hz_to_mel(hz))
    print(f'{hz} Hz ≈ {mel:.2f} Mel')
    print(f'转换回来：{float(mel_to_hz(mel)):.2f} Hz')
widgets.interact(mel_converter,hz=widgets.IntSlider(value=1000,min=0,max=8000,step=100,description='Hz'))

### 交互题 A

1. 分别记录 500、1000、2000、4000、8000 Hz 对应的 Mel。
2. Hz 每次翻倍时，Mel 是否也翻倍？
3. 在低频和高频区域，相同 100 Hz 增量造成的 Mel 增量哪边更大？

## 3. Mel 滤波器组是什么？

我们在 Mel 轴上等间距取点，再转换回 Hz。结果是在低频放置较密、高频放置较疏的一组三角形滤波器。每个滤波器把覆盖范围内的 FFT 功率加权求和。

$$E_m=\sum_k H_m[k]P[k]$$

$P[k]$ 是第 k 个 FFT bin 的功率，$H_m[k]$ 是第 m 个三角滤波器在该 bin 的权重。

In [ ]:
def mel_filterbank(sr,n_fft,n_mels=40,fmin=0,fmax=None):
    fmax=sr/2 if fmax is None else fmax
    mel_points=np.linspace(hz_to_mel(fmin),hz_to_mel(fmax),n_mels+2)
    hz_points=mel_to_hz(mel_points)
    fft_freqs=np.fft.rfftfreq(n_fft,1/sr)
    bank=np.zeros((n_mels,len(fft_freqs)))
    for m in range(n_mels):
        left,center,right=hz_points[m:m+3]
        rising=(fft_freqs-left)/max(center-left,1e-12)
        falling=(right-fft_freqs)/max(right-center,1e-12)
        bank[m]=np.maximum(0,np.minimum(rising,falling))
    return bank,fft_freqs,hz_points

bank,fft_freqs,hz_points=mel_filterbank(16000,512,40)
plt.figure(figsize=(12,5))
for filt in bank: plt.plot(fft_freqs,filt,linewidth=1)
plt.xlabel('Frequency (Hz)'); plt.ylabel('Weight')
plt.title('40 个 Mel 三角滤波器（16 kHz, n_fft=512）')
plt.grid(alpha=0.2); plt.show()

看图重点：每个频率通常会被附近两个滤波器以不同权重共同统计；低频三角形较窄，高频三角形较宽。滤波器不是把某个频率硬塞进唯一格子，而是平滑地汇总。

In [ ]:
def draw_filterbank(n_mels=40,n_fft=512):
    bank,ff,_=mel_filterbank(16000,n_fft,n_mels)
    plt.figure(figsize=(11,4))
    for row in bank: plt.plot(ff,row,linewidth=0.9)
    plt.xlabel('Frequency (Hz)'); plt.ylabel('Weight')
    plt.title(f'n_mels={n_mels}, n_fft={n_fft}, bank shape={bank.shape}')
    plt.grid(alpha=0.2); plt.show()
widgets.interact(draw_filterbank,
    n_mels=widgets.IntSlider(value=40,min=10,max=100,step=10,description='Mel bins'),
    n_fft=widgets.Dropdown(options=[256,512,1024,2048],value=512,description='n_fft'))

### 交互题 B

1. n_mels 从 20 增加到 80，三角形数量怎样变化？
2. n_fft 增大时，滤波器数量变吗？每个滤波器覆盖的 FFT bin 数量怎样变化？
3. 为什么高频滤波器通常比低频滤波器宽？

## 4. 一个滤波器如何汇总频谱？

下面生成 500、1500、3000 Hz 的混合信号，同时画功率谱和某几个 Mel 滤波器。滤波器与频谱相乘再求和，得到一个 Mel 能量。

In [ ]:
sr=16000
N=400
t=np.arange(N)/sr
x=np.sin(2*np.pi*500*t)+0.6*np.sin(2*np.pi*1500*t)+0.3*np.sin(2*np.pi*3000*t)
n_fft=512
power=np.abs(np.fft.rfft(x*np.hanning(N),n=n_fft))**2
bank,ff,_=mel_filterbank(sr,n_fft,40)
mel_energy=bank@power
fig,axes=plt.subplots(2,1,figsize=(12,7))
axes[0].plot(ff,power/power.max(),label='归一化功率谱')
for idx in [6,12,20,28]: axes[0].plot(ff,bank[idx],label=f'filter {idx}')
axes[0].set(xlabel='Frequency (Hz)',ylabel='Relative value',title='频谱与部分 Mel 滤波器')
axes[0].legend(); axes[0].grid(alpha=0.2)
axes[1].bar(np.arange(len(mel_energy)),mel_energy)
axes[1].set(xlabel='Mel bin',ylabel='Energy',title='40 个 Mel bin 的能量')
plt.tight_layout(); plt.show()

### 练习 1：看图题

1. 原始功率谱中主要有几个峰？
2. Mel 能量图是否仍有 257 个数？
3. 从 257 个 FFT bin 变成 40 个 Mel bin，是压缩还是扩展？
4. 这种压缩是否保留了每个精确 FFT bin？

## 5. 为什么还要取 Log？

频带能量可能跨越巨大范围。若最强能量为 10000，弱能量为 1，线性图中弱值几乎看不见。log 会压缩动态范围，更接近人耳对响度比例的感知，也更方便模型学习。

常见两种写法：`ln(E + eps)`，或用 dB 风格的 `10*log10(E/ref)`。不同工具实现可能不同，但核心都是对数压缩。

In [ ]:
energies=np.array([1,10,100,1000,10000],dtype=float)
natural_log=np.log(energies)
db=10*np.log10(energies/energies.max())
fig,axes=plt.subplots(1,3,figsize=(14,4))
axes[0].bar(range(len(energies)),energies); axes[0].set_title('线性能量')
axes[1].bar(range(len(energies)),natural_log); axes[1].set_title('自然对数 ln(E)')
axes[2].bar(range(len(energies)),db); axes[2].set_title('相对 dB（最大值=0 dB）')
for ax in axes: ax.set_xticks(range(len(energies)),[str(int(v)) for v in energies]); ax.grid(axis='y',alpha=0.2)
plt.tight_layout(); plt.show()

### 练习 2

1. 在线性图中，1 与 10 是否容易和 10000 同时看清？
2. 取 log 后，小能量是否更容易区分？
3. 相对 dB 中为什么最大值是 0 dB，其余通常是负数？

## 6. 从真实 WAV 得到 Log-Mel

流程：`WAV → frames → Hann → FFT → power → Mel filterbank → log`。本样本原始采样率为 8 kHz，因此最高频率是 4 kHz。

In [ ]:
audio,audio_sr=sf.read(ROOT/'data'/'0_jackson_0.wav')
if audio.ndim>1: audio=audio.mean(axis=1)
display(Audio(audio,rate=audio_sr))

def make_frames(audio,frame_size,hop_size):
    starts=np.arange(0,len(audio)-frame_size+1,hop_size)
    return np.stack([audio[s:s+frame_size] for s in starts]),starts

def stft_power(audio,sr,frame_ms=25,hop_ms=10,n_fft=256):
    frame_n=round(sr*frame_ms/1000); hop_n=round(sr*hop_ms/1000)
    frames,starts=make_frames(audio,frame_n,hop_n)
    z=np.fft.rfft(frames*np.hanning(frame_n)[None,:],n=n_fft,axis=1)
    return np.abs(z)**2,starts/sr,np.fft.rfftfreq(n_fft,1/sr)

power,times,freqs=stft_power(audio,audio_sr)
bank,_,_=mel_filterbank(audio_sr,256,40)
mel=power@bank.T
logmel=np.log(np.maximum(mel,1e-10))
print('STFT power shape:',power.shape)
print('Mel shape:',mel.shape)
print('Log-Mel shape:',logmel.shape)

In [ ]:
stft_db=10*np.log10(np.maximum(power,1e-12)); stft_db-=stft_db.max()
logmel_display=logmel-logmel.max()
fig,axes=plt.subplots(3,1,figsize=(12,10),sharex=True)
axes[0].plot(np.arange(len(audio))/audio_sr,audio,linewidth=0.7)
axes[0].set(title='Waveform',ylabel='Amplitude')
im1=axes[1].pcolormesh(times,freqs,stft_db.T,shading='auto',cmap='magma',vmin=-80,vmax=0)
axes[1].set(title='STFT spectrogram',ylabel='Frequency (Hz)')
im2=axes[2].pcolormesh(times,np.arange(40),logmel_display.T,shading='auto',cmap='viridis')
axes[2].set(title='Log-Mel spectrogram (40 bins)',xlabel='Time (s)',ylabel='Mel bin')
fig.colorbar(im1,ax=axes[1],label='Relative power (dB)')
fig.colorbar(im2,ax=axes[2],label='Relative log energy')
plt.tight_layout(); plt.show()

对比：STFT 纵轴是线性 Hz，频率行数为 129；Log-Mel 纵轴是 Mel bin，只有 40 行。低频细节被更多 Mel bin 表示，高频被更强地汇总。时间帧数保持不变。

## 7. 交互对比 20、40、80 个 Mel bins


In [ ]:
def draw_logmel(n_mels=40):
    bank,_,_=mel_filterbank(audio_sr,256,n_mels)
    mel=power@bank.T
    logmel=np.log(np.maximum(mel,1e-10)); logmel-=logmel.max()
    plt.figure(figsize=(11,4))
    im=plt.pcolormesh(times,np.arange(n_mels),logmel.T,shading='auto',cmap='viridis')
    plt.xlabel('Time (s)'); plt.ylabel('Mel bin')
    plt.title(f'Log-Mel: n_mels={n_mels}, shape={logmel.shape}')
    plt.colorbar(im,label='Relative log energy'); plt.show()
widgets.interact(draw_logmel,n_mels=widgets.IntSlider(value=40,min=10,max=100,step=10,description='Mel bins'))

### 交互题 C

1. n_mels 增大时，时间帧数变化吗？
2. 20 bins 与 80 bins 相比，哪个频率方向更细？
3. n_mels 越大是否一定越好？提示：细节、计算量、数据量和任务需求之间需要权衡。

## 8. 为什么 ASR 常用 Log-Mel，而不直接用 STFT？

- 更接近人耳的频率分辨方式。
- 从大量线性 FFT bin 压缩成较少 Mel bin。
- log 压缩巨大动态范围。
- 相比原始波形，对某些细微相位和幅度变化更稳定。
- 每一行保留‘某个感知频带在某个时间的能量’，很适合序列模型。

但现代模型也可以直接从波形学习前端。它们不是不需要特征，而是把特征提取交给网络学习。

## 本课测试

1. Mel 是物理频率单位还是感知频率尺度？
2. 低频和高频中，Mel 滤波器通常哪边更密？
3. Mel 滤波器为什么使用重叠三角形？
4. `bank @ power` 或 `power @ bank.T` 在做什么？
5. 从 257 个 FFT bin 变成 80 个 Mel bin，哪个维度改变？
6. 时间帧数会因为 Mel 滤波而改变吗？
7. 为什么取 log？
8. Log-Mel 的纵轴能直接读成 Hz 吗？
9. 16 kHz 音频的最高可用频率约是多少？
10. 40 个 Mel bin 和 80 个 Mel bin 的主要取舍是什么？
11. 写出完整流程：WAV 到 Log-Mel。
12. 为什么不能说 Mel ‘创造了新的频率信息’？

参考答案：1 感知尺度；2 低频；3 平滑汇总相邻频率，避免硬边界；4 将 FFT 功率按 Mel 频带加权求和；5 频率维；6 不会；7 压缩动态范围；8 不能，它是 Mel bin 编号；9 8 kHz；10 频率细节与计算量的权衡；11 分帧、加窗、FFT、功率、Mel 滤波、log；12 它只重组和压缩已有频谱信息。

## 小结

`STFT power (time × FFT bins) → Mel filterbank → Mel energy (time × Mel bins) → log → Log-Mel`

下一课：把整个声学前端封装成可复用函数，并与 librosa/torchaudio 等工具库的实现对照，检查 shape、单位、padding 和参数差异。

<!-- course-upgrade-v2 -->
## 强化练习：第 5 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `Mel 标度`、`滤波器组`、`Log-Mel`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**把 n_mels 从 40 改为 8 或 128**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**画出 Mel 三角滤波器并检查覆盖范围**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**解释 dB、log power 和模型输入的关系**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：Mel 标度、滤波器组、Log-Mel。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 Mel 标度、滤波器组、Log-Mel。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
